# Add Components To A Pipeline Config

The tutorial config is a normal JSON-compatible dictionary. Each component block has two parts:

- `type`: the registry name of the plugin implementation.
- `config`: the keyword arguments passed to that plugin's factory.

The standard PioneerML `training_pipeline` expects four step blocks: `hpo`, `train`, `evaluate`, and `export`. The tutorial config also includes an `inference` section to show where the model handle, writer, and output backend fit.

> **Notebook memory:** Importing the scientific Python stack (for example PyTorch, NumPy, Matplotlib, and ZenML) can keep approximately 1 GiB of RAM assigned to this kernel for its lifetime. Python cannot safely unload native extension modules. **After finishing this tutorial, restart its kernel to clear imported libraries and release that RAM** (or shut down the kernel entirely). Restarting keeps the notebook open with a fresh, low-memory kernel. Do this before running several tutorial notebooks at once.

In [ ]:
from pathlib import Path
import sys

# Notebooks are often opened from inside the package directory. This small
# bootstrap lets the same notebook work from a source checkout before the
# package is installed in editable mode.
for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    for src in (root / "src", root / "plugins" / "example_plugin" / "src"):
        if (src / "pioneerml_example_plugin").exists() or (src / "pioneerml").exists():
            src_text = str(src)
            if src_text not in sys.path:
                sys.path.insert(0, src_text)

from pioneerml_example_plugin.components_tutorial_examples.pipeline import load_config

full_config = load_config()
training_config = full_config["training"]
print(full_config.keys())
print(training_config.keys())

In [ ]:
for step in ["hpo", "train", "evaluate", "export"]:
    print(f"training.{step}")
    for block, value in training_config[step].items():
        if isinstance(value, dict) and "type" in value:
            print(f"  {block:15s} -> {value['type']}")

print("inference.model_handle ->", full_config["inference"]["model_handle_builder"]["model_handle"]["type"])
print("inference.executor     ->", full_config["inference"]["inference"]["batch_executor"]["type"])
print("inference.writer       ->", full_config["inference"]["inference"]["writer"]["type"])

## Loader Wiring

Each pipeline step owns a loader-manager config. This tutorial points `input_sources_spec.main_sources` at `sample_data/sensor_health.csv`, which is bundled with the plugin.

There is no separate data block because the input source spec is the source of truth. The loader manager resolves the package-local path before core PioneerML validates that the file exists.

In [ ]:
training_config["train"]["loader_manager"]

## Training Wiring

The train step reads these blocks:

- `architecture`: builds the PyTorch graph model.
- `compiler`: optionally wraps or optimizes the model.
- `module`: wraps the model and loss in a LightningModule.
- `trainer`: controls the Lightning training loop.
- `loader_manager`: builds train and validation dataloaders.

The HPO step uses the same model/trainer/loader shape, plus an `hpo` block that names the objective and search space.

In [ ]:
{key: training_config["train"][key] for key in ["architecture", "compiler", "module", "trainer"]}